# Melanoma graph classification task {-}

This notebook contains an interactive workflow for the melanoma patient manifold classification experiments (the random bandlimited signal and individual eigenfunction learning tasks). Use it to (1) generate melanoma patient datasets, (2) run k-folds cross-validation for MFCN and baseline models, and (3) summarize results in tables and plots, for both tasks.

> **Important**: to set up directories for the project based on a string key for the system in use, set the `MACHINE` keys (e.g., "desktop") and `ROOT` path (to the project folder on your "desktop" machine) args in the `__post_init__` method of `args_template.py`. Then, use your chosen `MACHINE` key when initializing `args` in this first cell and in script calls with the `--machine` argument.

In [ ]:
import sys
sys.path.insert(0, '../')
import pandas as pd
import results_utilities as ru
import melanoma.args as a

args = a.Args(MACHINE="desktop")

## Dataset creation {-}

This script processes the 3 melanoma MIBI data CSV files into a pytorch-geometric dataset, in train/valid set splits, and saves into your `data/` directory (you only need to run it once before training models in the next section of this notebook). To run this script:

- These 3 CSV files must be in the directory set in `melanoma.args.DATA_DIR`.
- Make sure you have set the correct paths for each `--machine` flag in `args_template.py`.
- You may have to provide the absolute path to `create_melanoma_pyg_dataset.py` file in the cell below.
- Make sure `python3` is calling the correct version of python (>=3.11), and that it is available as an ipykernel to Jupyter.

In [ ]:
! python3.11 "../melanoma/create_melanoma_pyg_dataset.py" \
--machine "desktop"

## Model training (k-folds CV) {-}

- You may have to provide the absolute path to `experiments_cv.py` in the cell below.

**Include model flags in the command-line arguments for the model types you wish to fit:**

- MFCN-low-pass-spectral filter model: set the value of $t$ as the `SPECTRAL_C` argument in melanoma/args.py file. Flag: `--mcn_spectral`

- MFCN-low-pass-approx (diffusion) filter model: `--mcn_p`

- MFCN with spectral wavelets: `--mfcn_spectral`

- MFCN with dyadic diffusion (approx) wavelets: `--mfcn_p`

- MFCN with 'Infogain' diffusion (approx) wavelets: `--mfcn_p --smart_p_wavelets`

- Baseline GNNs: `--gcn --gat --sage --gin`

In [ ]:
! python3.11 "../train_scripts/experiments_cv.py" \
--machine "desktop" \
--mcn_spectral \
--spectral_c 0.5 \
--mcn_within_filter_chan_out 32,16 \
--smart_p_wavelets \
--dataset "melanoma" \
--n_folds 10 \
--n_epochs 1000 \
--burn_in 100 \
--patience 50 \
--learn_rate 0.005 \
--batch_size 8 \
--verbosity 0

## Results {-}

After running cross-validations on the desired models, add the paths to their results folders to the `model_dirs: Tuple[str]` variable in this block:

#### Results of each model's cross-validations runs {-}

In [ ]:
root = "{ROOT_DIR}"
# args.MODEL_SAVE_DIR = "{NEW_DIR}"
# args.RESULTS_SAVE_DIR = "{NEW_DIR}"

# model_dirs: tuple of model directories to load results from
model_dirs = (
    ""
)

# model_suffix_dict: dict of model suffixes to display in the results table
model_suffix_dict = {
    "": ""
}

results_df = ru.get_cv_results_df(
    args, 
    model_dirs, 
    include_times=True,
    model_suffix_dict=model_suffix_dict,
    decimal_round=4
).sort_values('model')

# inspect results
results_df

#### Summary of all cross-validation runs, by model (mean of means; mean st. dev.) {-}

In [ ]:
# average across [5] reps x 10-fold CVs
summary_df = ru.avg_avg_results_df(results_df) # .round(col_rounds)

# set decimal rounding settings by (multilevel) cols
col_rounds = {
    ('acc', 'mean', 'nanmean'): 2,
    ('acc', 'std', 'mean_std'): 2,
    ('specificity', 'mean', 'nanmean'): 2,
    ('specificity', 'std', 'mean_std'): 2,
    ('f1', 'mean', 'nanmean'): 2,
    ('f1', 'std', 'mean_std'): 2,
    ('f1_neg', 'mean', 'nanmean'): 2,
    ('f1_neg', 'std', 'mean_std'): 2,
    ('sec_per_epoch', 'mean', 'nanmean'): 4,
    ('sec_per_epoch', 'std', 'mean_std'): 4,
    ('n_epochs', 'mean', 'nanmean'): 0,
    ('n_epochs', 'std', 'mean_std'): 0,
    ('train_min_per_fold', 'mean', 'nanmean'): 2,
    ('train_min_per_fold', 'std', 'mean_std'): 2
}

# set final col names mapping
final_colnames = {
    'acc': 'Accuracy',
    'f1': 'F1 score',
    'sec_per_epoch': 'Sec. per epoch',
    'n_epochs': 'Num. epochs',
    'train_min_per_fold': 'Min. per fold'
}

# generate final summary table and its latex string
summary_df, df_latex = ru.get_mean_pm_std_df(
    df=summary_df,
    mean_std_colnames=('acc', 'f1', 'sec_per_epoch', 'n_epochs', 'train_min_per_fold'),
    col_rounds=col_rounds,
    final_colnames=final_colnames,
    mean_subcol_tuple=('mean', 'nanmean'),
    std_subcol_tuple=('std', 'mean_std'),
    sort_col_tuple=('acc', 'mean', 'nanmean'),
    sort_col_ascends=[False],
    best_bolding_fns_dict=None
)

# inspect results
print(df_latex)
summary_df